# 07 – Room Graph

Detects enclosed rooms from wall surfaces using `CellComplex.ByFaces`,
then builds a proper room-to-room adjacency and access graph.

- 1 node = 1 room
- 1 edge = shared wall (adjacency) or shared door/window (access)

References: S02-03, S02-05

## 1. Import libraries

In [ ]:
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from pathlib import Path

## 2. Check version

In [ ]:
print("Requires topologicpy 0.9.18 or newer.")
print(Helper.Version())

## 3. Set renderer

In [ ]:
renderer = "vscode"

## 4. Load the geometry OBJ

In [ ]:
base = Path("c:/Users/etmaglari/IAAC/etmaglari_gML")

geo_path = next((p for p in [
    Path.cwd() / "Rhino_Geometry _etm.obj",
    base / "Rhino_Geometry _etm.obj",
] if p.exists()), None)

if geo_path is None:
    raise FileNotFoundError("Rhino_Geometry _etm.obj not found.")

geo_objects = Topology.ByOBJPath(str(geo_path))
print(f"Loaded {len(geo_objects)} object(s) from {geo_path.name}")

## 5. Extract all faces from the mesh

Each loaded object is a mesh. We pull every face out of every object
to give `CellComplex.ByFaces` the full set of surfaces to work with.

In [ ]:
all_faces = []
for obj in geo_objects:
    all_faces.extend(Topology.Faces(obj))

print(f"Total faces extracted: {len(all_faces)}")

## 6. Build a CellComplex — detect rooms

`CellComplex.ByFaces` finds all enclosed volumes in the face soup.
Each `Cell` in the result = one room.
If tolerance is too tight, increase it (0.01 → 0.1 → 1.0).

*Ref: S02-03 § 4*

In [ ]:
cellcomplex = None
for tol in [0.01, 0.1, 1.0]:
    cellcomplex = CellComplex.ByFaces(all_faces, tolerance=tol)
    if cellcomplex is not None:
        cells = Topology.Cells(cellcomplex)
        print(f"Tolerance {tol}: {len(cells)} room(s) detected.")
        break

if cellcomplex is None:
    print("CellComplex failed. The wall surfaces may not form fully enclosed volumes.")
    print("Check that all walls meet cleanly with no gaps in Rhino.")

## 7. Show the detected rooms

In [ ]:
from topologicpy.Color import Color

cells = Topology.Cells(cellcomplex)
colors = Color.ColorRamp(len(cells))

for i, cell in enumerate(cells):
    d = Dictionary.ByKeysValues(["color", "room_id"], [colors[i], i])
    cell = Topology.SetDictionary(cell, d)

Topology.Show(cellcomplex,
              faceColorKey="color",
              faceOpacity=0.4,
              edgeColor="white",
              edgeWidth=1,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 8. Build the adjacency graph

One node per room. Two rooms are connected if they share a wall face.

*Ref: S02-03 § 6–8, S02-05 § 6–8*

In [ ]:
g_adjacency = Graph.ByTopology(cellcomplex)
print(f"Adjacency graph: {len(Graph.Vertices(g_adjacency))} rooms, {len(Graph.Edges(g_adjacency))} connections.")

for v in Graph.Vertices(g_adjacency):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [15, "red"]))
for e in Graph.Edges(g_adjacency):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [3, "white"]))

Topology.Show(cellcomplex, g_adjacency,
              faceOpacity=0.05,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 9. Load apertures and build the access graph

The access graph only connects rooms that share a door or window —
not every shared wall.

*Ref: S02-03 § 9–11, S02-05 § 9–11*

In [ ]:
apt_path = next((p for p in [
    Path.cwd() / "Rhino_Apertures _etm.obj",
    base / "Rhino_Apertures _etm.obj",
] if p.exists()), None)

if apt_path is None:
    raise FileNotFoundError("Rhino_Apertures _etm.obj not found.")

apt_objects = Topology.ByOBJPath(str(apt_path))
aperture_faces = []
for obj in apt_objects:
    aperture_faces.extend(Topology.Faces(obj))

print(f"Aperture faces: {len(aperture_faces)}")

cc_with_apts = Topology.AddApertures(cellcomplex, aperture_faces, subTopologyType="face")

g_access = Graph.ByTopology(cc_with_apts, direct=False, viaSharedApertures=True)
print(f"Access graph: {len(Graph.Vertices(g_access))} rooms, {len(Graph.Edges(g_access))} connections.")

## 10. Show the access graph

In [ ]:
for v in Graph.Vertices(g_access):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [15, "red"]))
for e in Graph.Edges(g_access):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [3, "#2CFF96"]))

Topology.Show(cc_with_apts, g_access,
              faceOpacity=0.05,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)